# Lección 6: Eliminación Gaussiana e Ingeniería de Precisión Finita
### Álgebra Lineal Computacional y Fundamentos de la Física Matemática

---

Este cuaderno de Jupyter Colab establece los fundamentos teóricos y computacionales de los **métodos directos de resolución** de sistemas de ecuaciones lineales, con énfasis en el algoritmo de **Eliminación Gaussiana**. Se estudian los efectos de la aritmética de precisión finita en ordenadores, la propagación de errores de redondeo, la evaluación de residuos y errores, y las estrategias de mitigación mediante **pivotaje parcial** y **pivotaje parcial escalado**. El tratamiento combina el rigor analítico formal con implementaciones en Python para verificar y visualizar los algoritmos.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Comprender** la representación matricial de sistemas de ecuaciones lineales $AX = B$ y clasificar las soluciones según el teorema de Rouché-Frobenius.
2. **Reconocer** las diferencias estructurales y de aplicación entre métodos de resolución directos e iterativos.
3. **Implementar** el algoritmo de Eliminación Gaussiana Ingenua (Naive) y la sustitución regresiva.
4. **Evaluar** la calidad de soluciones numéricas calculando el vector de error y el vector residual.
5. **Comprender** la inestabilidad numérica que causan los pivotes cercanos a cero debido a la aritmética de coma flotante de precisión finita.
6. **Programar y aplicar** las técnicas de pivotaje parcial y pivotaje parcial escalado para estabilizar la eliminación gaussiana.


## 1. Resolución Automática de Sistemas de Ecuaciones

Resolver un sistema de ecuaciones lineales de la forma:
$$A X = b$$
donde $A \in \mathbb{R}^{m \times n}$ es la matriz de coeficientes, $X \in \mathbb{R}^n$ es el vector de incógnitas y $b \in \mathbb{R}^m$ es el vector de términos independientes, es una de las tareas más comunes en la física y la ingeniería.

### 1.1 Representación Geométrica en Dos Dimensiones
En 2D ($m=n=2$), cada ecuación representa una recta en el plano cartesiano.
* Si las rectas se interceptan en un único punto, el sistema es **Compatible Determinado** (solución única).
* Si las rectas son paralelas y distintas, el sistema es **Incompatible** (sin solución).
* Si las rectas son coincidentes (la misma recta), el sistema es **Compatible Indeterminado** (infinitas soluciones).

### 1.2 Clasificación Lógica de Sistemas (Teorema de Rouché-Frobenius)
El número de soluciones se determina a partir del rango de la matriz de coeficientes $A$ y de la matriz aumentada $\tilde{A} = (A \mid b)$:
* **Sistema Compatible (tiene solución):** $\text{rango}(A) = \text{rango}(A \mid b)$.
  * **Determinado (solución única):** $\text{rango}(A) = n$ (donde $n$ es el número de incógnitas).
  * **Indeterminado (infinitas soluciones):** $\text{rango}(A) < n$.
* **Sistema Incompatible (no tiene solución):** $\text{rango}(A) \neq \text{rango}(A \mid b)$.

#### Equivalencia para Matrices Cuadradas ($n \times n$)
Si $A$ es cuadrada, las siguientes afirmaciones son equivalentes:
1. $\text{rango}(A) = n$.
2. El núcleo (kernel) de $A$ es trivial: $\text{ker}(A) = \{0\}$.
3. Las columnas (y filas) de $A$ son linealmente independientes.
4. La matriz $A$ es invertible (existe $A^{-1}$).
5. El determinante de $A$ es distinto de cero ($\det(A) \neq 0$).

### 1.3 Métodos Directos vs. Iterativos
* **Métodos Directos (ej. Eliminación Gaussiana):** Transforman el sistema en uno equivalente cuya solución se calcula en un número **finito** y predecible de pasos. Teóricamente proporcionan la solución exacta (salvo errores de redondeo).
* **Métodos Iterativos (ej. Jacobi, Gauss-Seidel):** Generan una secuencia de aproximaciones sucesivas que convergen hacia la solución. El proceso puede detenerse en cualquier iteración al alcanzar la tolerancia deseada.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Graficamos el sistema de la Figura 1 del PDF:
# x - y = 1 => y = x - 1
# x + y = 3 => y = 3 - x
# La solución de intersección es (2, 1)

x = np.linspace(0, 4, 200)
y1 = x - 1
y2 = 3 - x

plt.figure(figsize=(8, 6))
plt.plot(x, y1, 'r-', label='Recta 1: $x - y = 1$')
plt.plot(x, y2, 'b-', label='Recta 2: $x + y = 3$')
plt.scatter(2, 1, color='black', s=100, zorder=5, label='Interseccion (2, 1)')

plt.title('Representacion Geometrica de un Sistema 2D', fontsize=13, fontweight='bold')
plt.xlabel('Eje X', fontsize=11)
plt.ylabel('Eje Y', fontsize=11)
plt.axhline(0, color='gray', lw=0.8)
plt.axvline(0, color='gray', lw=0.8)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()



## 2. Eliminación Gaussiana Ingenua (Naive)

La eliminación gaussiana consiste en transformar un sistema general $AX = b$ en un sistema triangular superior equivalente $UX = b'$ mediante operaciones elementales de fila:
1. Multiplicación de una fila por una constante no nula.
2. Sumar a una fila otra fila multiplicada por un escalar.
3. Permutación de dos filas.

El término **"ingenua"** se refiere a que se asume que las entradas de la diagonal (pivotes) son siempre distintas de cero, y se realiza la reducción de forma secuencial sin alterar el orden original de las filas.

### 2.1 Fase de Eliminación Hacia Adelante (Forward Elimination)
Para cada columna $k$ desde $1$ hasta $n-1$:
Para cada fila $i$ desde $k+1$ hasta $n$:
1. Calcular el multiplicador:
   $$m_{ik} = \frac{a_{ik}}{a_{kk}}$$
2. Restar la fila $k$ multiplicada por $m_{ik}$ a la fila $i$:
   $$F_i \leftarrow F_i - m_{ik} \cdot F_k$$

### 2.2 Fase de Sustitución Regresiva (Backward Substitution)
Una vez que el sistema se ha reducido a la forma triangular superior, las incógnitas se despejan en orden inverso (de $n$ a 1):
$$x_n = \frac{b'_n}{u_{nn}}$$
$$x_k = \frac{1}{u_{kk}} \left( b'_k - \sum_{i=k+1}^n u_{ki} \cdot x_i \right) \quad \text{para } k = n-1, n-2, \dots, 1$$


In [ ]:
# 1. Implementación de la Eliminación Gaussiana Ingenua
def eliminacion_gaussiana_ingenua(A, b):
    # A es una matriz de floats, b es un vector columna de floats
    n = len(b)
    # Copiamos para no modificar los originales
    Ab = np.column_stack((A.astype(float), b.astype(float)))
    
    # Eliminación hacia adelante
    for k in range(n - 1):
        for i in range(k + 1, n):
            if Ab[k, k] == 0:
                raise ValueError("Pivote nulo detectado. El metodo ingenuo falla.")
            multiplicador = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= multiplicador * Ab[k, k:]
            
    # Sustitución regresiva
    x = np.zeros(n)
    x[n - 1] = Ab[n - 1, n] / Ab[n - 1, n - 1]
    for k in range(n - 2, -1, -1):
        suma = np.dot(Ab[k, k + 1:n], x[k + 1:n])
        x[k] = (Ab[k, n] - suma) / Ab[k, k]
        
    return x

# 2. Ejemplo 3 del PDF (Página 7)
# 2x + 3y + z = 0
# x + y + 2z = 1
# x - y - z = -1
A_test = np.array([
    [2.0, 3.0, 1.0],
    [1.0, 1.0, 2.0],
    [1.0, -1.0, -1.0]
])
b_test = np.array([0.0, 1.0, -1.0])

sol = eliminacion_gaussiana_ingenua(A_test, b_test)
print("Solucion del Ejemplo 3:")
print(f"  x = {sol[0]:.4f} (esperado: -1/3 = -0.3333)")
print(f"  y = {sol[1]:.4f} (esperado: 0.0000)")
print(f"  z = {sol[2]:.4f} (esperado: 2/3 = 0.6667)")



## 3. Aritmética de Precisión Finita y Errores de Redondeo

En un ordenador físico, los números reales se representan con **precisión finita** (típicamente precisión doble de 64 bits siguiendo el estándar IEEE 754). Esto significa que cada operación matemática introduce un pequeño error de redondeo.

### 3.1 El Problema del Pivote Pequeño
Si un coeficiente de la diagonal principal (pivote $a_{kk}$) es muy cercano a cero, el multiplicador $m_{ik} = a_{ik}/a_{kk}$ se vuelve muy grande. Al multiplicar la fila por este factor y restarla de otra, los errores de redondeo pequeños se magnifican, destruyendo la precisión de la solución.

### 3.2 Vector de Error y Vector Residual
Para evaluar la calidad de una solución calculada $\tilde{X}$ frente a la solución exacta $X$:
* **Vector de Error ($e$):** La diferencia directa entre la solución exacta y la numérica:
  $$e = X - \tilde{X}$$
* **Vector Residual ($r$):** Mide qué tan bien la solución numérica satisface las ecuaciones originales:
  $$r = b - A \tilde{X}$$
  *(Nota: En sistemas inestables o mal condicionados, un residuo pequeño no garantiza un error pequeño).*

### 3.3 Simulación del Error Crítico del PDF (Página 10)
Consideremos el siguiente sistema lineal resuelto con **redondeo a 4 dígitos significativos**:
$$0.0002 x_1 + 1.471 x_2 = 1.473$$
$$0.2346 x_1 - 1.317 x_2 = 1.029$$
La solución exacta es $x_1 = 10, x_2 = 1$.

* **Ejecución Ingenua (Sin Pivoteo):**
  * El pivote es $a_{11} = 0.0002$.
  * Multiplicador: $m_{21} = 0.2346 / 0.0002 = 1173$.
  * Al realizar las restas de fila con redondeo de 4 dígitos:
    * $a_{22} = -1.317 - 1173 \times 1.471 = -1.317 - 1725 \approx -1726$.
    * $b_2 = 1.029 - 1173 \times 1.473 = 1.029 - 1728 \approx -1727$.
  * Despejamos:
    * $-1726 x_2 = -1727 \implies x_2 \approx 1.001$.
    * Sustituyendo atrás:
      $$0.0002 x_1 + 1.471(1.001) = 1.473 \implies 0.0002 x_1 + 1.472 = 1.473 \implies 0.0002 x_1 = 0.001 \implies x_1 = 5.000$$
  * La solución calculada es $x_1 = 5.000$ (¡un error relativo del 50%!) debido a la pequeñez del pivote original.


In [ ]:
# 1. Función para simular aritmética de redondeo a d dígitos significativos
def redondear_d_digitos(x, d=4):
    if x == 0:
        return 0.0
    # Obtenemos el exponente
    exp = int(np.floor(np.log10(abs(x))))
    # Redondeamos la mantisa
    factor = 10**(d - 1 - exp)
    return np.round(x * factor) / factor

# Vectorizamos la función para aplicar a matrices
redondear_vec = np.vectorize(redondear_d_digitos)

# 2. Simulación de la Eliminación Ingenua con 4 dígitos de redondeo
print("--- Simulacion de Eliminacion Ingenua (4 digitos) ---")
a11, a12, b1 = 0.0002, 1.471, 1.473
a21, a22, b2 = 0.2346, -1.317, 1.029

# Multiplicador redondeado
m21 = redondear_d_digitos(a21 / a11)
print(f"Multiplicador m21 = {m21}")

# Fila 2 modificada con redondeo
a22_nueva = redondear_d_digitos(a22 - redondear_d_digitos(m21 * a12))
b2_nueva = redondear_d_digitos(b2 - redondear_d_digitos(m21 * b1))
print(f"Fila 2 modificada: {a22_nueva}*x2 == {b2_nueva}")

# Despeje de x2 con redondeo
x2_calc = redondear_d_digitos(b2_nueva / a22_nueva)
# Despeje de x1 con redondeo
x1_calc = redondear_d_digitos(redondear_d_digitos(b1 - redondear_d_digitos(a12 * x2_calc)) / a11)

print(f"Solucion calculada: x1 = {x1_calc}, x2 = {x2_calc}")

# 3. Cálculo de Error y Residuo
x_real = np.array([10.0, 1.0])
x_calc = np.array([x1_calc, x2_calc])
A_mat = np.array([[a11, a12], [a21, a22]])
b_vec = np.array([b1, b2])

error_vec = x_real - x_calc
residuo_vec = b_vec - A_mat @ x_calc

print(f"Vector de Error e: {error_vec}")
print(f"Vector Residual r: {residuo_vec}")
print(f"Error relativo de x1: {abs(error_vec[0])/x_real[0] * 100:.1f}%")



## 4. Eliminación Gaussiana con Pivotaje

Para mitigar los errores catastróficos causados por pivotes nulos o pequeños, reordenamos las filas antes de cada paso de eliminación.

### 4.1 Pivotaje Parcial
Consiste en comparar todos los elementos de la columna activa $k$ desde la diagonal hacia abajo. Intercambiamos la fila $k$ con la fila $p$ que contiene el elemento de mayor valor absoluto:
$$|a_{pk}| = \max_{k \le i \le n} |a_{ik}|$$
* En el ejemplo crítico anterior, al buscar en la columna 1:
  $$|a_{21}| = 0.2346 > |a_{11}| = 0.0002$$
  El algoritmo intercambia la Fila 1 y la Fila 2. Al resolver el sistema reordenado bajo el mismo redondeo de 4 dígitos, se obtiene la solución exacta $x_1 = 10, x_2 = 1$.

### 4.2 Pivotaje Parcial Escalado
El pivotaje parcial simple puede fallar si los elementos de una fila son desproporcionadamente grandes comparados con los de otras filas. El **pivotaje parcial escalado** corrige esto dividiendo los candidatos de la columna por el factor de escala de su propia fila.

1. **Factor de Escala ($s_i$):** El elemento de mayor valor absoluto en cada fila $i$:
   $$s_i = \max_{1 \le j \le n} |a_{ij}|$$
2. **Selección del Pivote:** En el paso $k$, se elige la fila $p$ que maximiza la relación relativa:
   $$\frac{|a_{pk}|}{s_p} = \max_{k \le i \le n} \frac{|a_{ik}|}{s_i}$$
3. Se intercambian las filas y se procede con el paso habitual de eliminación.


In [ ]:
# 1. Simulación del Sistema Crítico con Pivotaje Parcial y Redondeo
print("--- Solucion con Pivotaje Parcial (4 digitos) ---")
# Intercambiamos las filas
a11_p, a12_p, b1_p = 0.2346, -1.317, 1.029
a21_p, a22_p, b2_p = 0.0002, 1.471, 1.473

# Multiplicador redondeado
m21_p = redondear_d_digitos(a21_p / a11_p)
print(f"Nuevo multiplicador m21 = {m21_p}")

# Fila 2 modificada
a22_p_nueva = redondear_d_digitos(a22_p - redondear_d_digitos(m21_p * a12_p))
b2_p_nueva = redondear_d_digitos(b2_p - redondear_d_digitos(m21_p * b1_p))
print(f"Fila 2 modificada: {a22_p_nueva}*x2 == {b2_p_nueva}")

# Despejes
x2_p_calc = redondear_d_digitos(b2_p_nueva / a22_p_nueva)
x1_p_calc = redondear_d_digitos(redondear_d_digitos(b1_p - redondear_d_digitos(a12_p * x2_p_calc)) / a11_p)

print(f"Solucion con Pivotaje: x1 = {x1_p_calc}, x2 = {x2_p_calc}")
print("-" * 50)


# 2. Implementación de la Eliminación Gaussiana con Pivotaje Parcial Escalado
def gauss_pivotaje_parcial_escalado(A, b):
    n = len(b)
    Ab = np.column_stack((A.astype(float), b.astype(float)))
    
    # Calcular los factores de escala iniciales para cada fila
    s = np.zeros(n)
    for i in range(n):
        s[i] = np.max(np.abs(Ab[i, :n]))
        if s[i] == 0:
            raise ValueError(f"Fila {i} nula detectada. El sistema no tiene solucion unica.")
            
    # Eliminación hacia adelante con pivotaje escalado
    for k in range(n - 1):
        # Buscar el pivote
        relacion_max = -1
        fila_pivote = k
        for i in range(k, n):
            relacion = np.abs(Ab[i, k]) / s[i]
            if relacion > relacion_max:
                relacion_max = relacion
                fila_pivote = i
                
        # Intercambio de filas si es necesario
        if fila_pivote != k:
            Ab[[k, fila_pivote]] = Ab[[fila_pivote, k]]
            s[[k, fila_pivote]] = s[[fila_pivote, k]]
            
        # Eliminación
        for i in range(k + 1, n):
            multiplicador = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= multiplicador * Ab[k, k:]
            
    # Sustitución regresiva
    x = np.zeros(n)
    x[n - 1] = Ab[n - 1, n] / Ab[n - 1, n - 1]
    for k in range(n - 2, -1, -1):
        suma = np.dot(Ab[k, k + 1:n], x[k + 1:n])
        x[k] = (Ab[k, n] - suma) / Ab[k, k]
        
    return x

# Probamos con el mismo sistema 3x3 anterior
sol_escalado = gauss_pivotaje_parcial_escalado(A_test, b_test)
print("Solucion con Pivotaje Parcial Escalado (Ejemplo 3):")
print("  x =", sol_escalado)



## 5. Resumen de la Lección

1. **Sistemas de Ecuaciones:** Se clasifican según el Teorema de Rouché-Frobenius mediante la comparación del rango de la matriz de coeficientes y de la matriz aumentada.
2. **Método Directo:** La eliminación de Gauss reduce un sistema a su forma triangular superior mediante operaciones elementales de fila en un número finito de pasos.
3. **Propagación del Redondeo:** En la computación real de precisión finita, operar con pivotes muy pequeños en relación al resto de coeficientes propaga errores que distorsionan la solución final.
4. **Validación:** El residuo ($r = b - AX$) y el error ($e = X_{real} - X_{calc}$) son herramientas complementarias para auditar la precisión del cálculo.
5. **Estrategias de Pivotaje:** El pivotaje parcial (simple o escalado) evita la división por cero y minimiza la propagación de errores numéricos de redondeo al seleccionar el elemento con mayor magnitud de la columna.

---

## 6. Referencias Bibliográficas

1. del Valle Sotelo, J. C. (2012). *Álgebra lineal para estudiantes de ingeniería y ciencias*. Mc Graw Hill.
2. O’Connor, J. J. (1997). *Técnicas de Cálculo para Sistemas de Ecuaciones, Programación Lineal y Programación Entera*. Reverte, Escuela Técnica Superior de Ingenieros Industriales, p. 966.
3. Quezada, R. (2019). *Métodos Directos para resolver Sistemas de Ecuaciones Lineales*. Cap. 2, p. 16.
4. ma2008 (2009). *Métodos Iterativos para Resolver Sistemas Lineales*.
5. González, I. *Resolución de sistemas de ecuaciones lineales. Métodos directos*. Escuela Técnica Superior de Ingeniería de Telecomunicación.
6. *Método de Eliminación de Gauss*. (2020). Problemas y Ecuaciones. Recuperado de: https://www.problemasyecuaciones.com/matrices/metodoeliminacion-gauss-jordan-matrices-sistemas-ecuaciones-lineales-resueltos-ejemplos-matriz.html
7. Zapata, M. (2014). *Eliminación Gaussiana con Pivotaje y Matrices de Banda*. Facultad de Matemáticas, UADY.
